# 04 — Robustness Checks

Deflated Sharpe Ratio, regime analysis, parameter sensitivity, walk-forward OOS.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dataclasses

from src.config import load_config, StrategyConfig
from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.universe import UniverseProvider
from src.features.registry import build_features
from src.signals.combiner import SignalCombiner
from src.portfolio.constructor import PortfolioConstructor
from src.backtest.engine import BacktestEngine
from src.backtest.walkforward import WalkForwardOptimizer
from src.risk.manager import RiskManager
from src.analytics.performance import compute_metrics
from src.analytics.statistics import deflated_sharpe_ratio, lo_adjusted_sharpe_test

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load data (reuse from notebook 03)
cfg = load_config('../config.yaml')
universe = UniverseProvider(cfg.data)
tickers = universe.get_tickers()

loader = DataLoader(cfg.data)
raw_data = loader.load_universe(tickers)
prices = loader.build_price_matrix(raw_data)
volumes = loader.build_volume_matrix(raw_data)

valid_tickers = universe.filter_universe(prices, volumes)
prices = prices[valid_tickers]

cleaner = DataCleaner(cfg.data)
prices, returns = cleaner.clean(prices, volumes)

In [ ]:
def run_strategy(cfg, prices, returns):
    """Run full strategy pipeline and return BacktestResult."""
    features = build_features(cfg.features)
    signal_dict = {}
    for f in features:
        signal_dict[f.name] = f.compute(prices, returns)
    
    combiner = SignalCombiner(cfg.signals)
    combined = combiner.combine(signal_dict, returns)
    
    constructor = PortfolioConstructor(cfg.portfolio)
    weights = constructor.construct(combined, returns)
    
    risk_mgr = RiskManager(cfg.risk)
    adjusted = risk_mgr.apply(weights, returns)
    
    engine = BacktestEngine(cfg.backtest, cfg.execution)
    return engine.run(adjusted, returns)

In [ ]:
# 1. Deflated Sharpe Ratio Analysis
# Run baseline strategy
result = run_strategy(cfg, prices, returns)
metrics = compute_metrics(result.net_returns)

# DSR with different trial counts
print('=== Deflated Sharpe Ratio Analysis ===')
print(f'Observed Sharpe: {metrics.sharpe_ratio:.3f}')
print()

for n_trials in [1, 3, 6, 10, 20, 50]:
    dsr = deflated_sharpe_ratio(
        sharpe_observed=metrics.sharpe_ratio,
        n_trials=n_trials,
        n_obs=len(result.net_returns),
        skewness=metrics.skewness,
        kurtosis=metrics.kurtosis + 3,
    )
    print(f'N={n_trials:3d} trials: DSR={dsr["dsr"]:.4f}, '
          f'E[max SR]={dsr["expected_max_sharpe"]:.3f}, '
          f'p={dsr["p_value"]:.4f}')

In [ ]:
# 2. Lo-Adjusted Sharpe Test
lo = lo_adjusted_sharpe_test(result.net_returns, cfg.analytics.risk_free_rate)
print('\n=== Lo (2002) Autocorrelation-Adjusted Sharpe ===')
print(f'Raw Sharpe:      {lo["sharpe"]:.3f}')
print(f'Adjusted Sharpe: {lo["adjusted_sharpe"]:.3f}')
print(f'Eta (autocorr):  {lo["eta"]:.3f}')
print(f't-statistic:     {lo["t_stat"]:.3f}')
print(f'p-value:         {lo["p_value"]:.4f}')

In [ ]:
# 3. Parameter sensitivity: vary key parameters
print('\n=== Parameter Sensitivity ===')

# Vary momentum lookback
sensitivity_results = []
for skip in [5, 10, 21, 42, 63]:
    mod_cfg = dataclasses.replace(
        cfg,
        features=dataclasses.replace(
            cfg.features,
            momentum=dataclasses.replace(cfg.features.momentum, xsmom_skip=skip)
        )
    )
    res = run_strategy(mod_cfg, prices, returns)
    m = compute_metrics(res.net_returns)
    sensitivity_results.append({
        'Parameter': 'xsmom_skip',
        'Value': skip,
        'Sharpe': m.sharpe_ratio,
        'Return': m.annual_return,
        'MaxDD': m.max_drawdown,
    })

sens_df = pd.DataFrame(sensitivity_results)
print(sens_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# 4. Parameter sensitivity plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(sens_df['Value'].astype(str), sens_df['Sharpe'], color='steelblue')
axes[0].set_title('Sharpe vs XSMOM Skip Period')
axes[0].set_xlabel('Skip Days')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].axhline(y=0, color='black', linewidth=0.5)

axes[1].bar(sens_df['Value'].astype(str), sens_df['MaxDD'] * 100, color='indianred')
axes[1].set_title('Max Drawdown vs XSMOM Skip Period')
axes[1].set_xlabel('Skip Days')
axes[1].set_ylabel('Max Drawdown (%)')

plt.tight_layout()
plt.show()

In [ ]:
# 5. Walk-forward OOS analysis
print('\n=== Walk-Forward Out-of-Sample Analysis ===')
wf = WalkForwardOptimizer(cfg.backtest.walk_forward)
windows = wf.generate_windows(len(prices))
print(f'Generated {len(windows)} walk-forward windows')

is_sharpes = []
oos_sharpes = []

for i, window in enumerate(windows):
    # In-sample
    is_prices = prices.iloc[window.train_start:window.train_end]
    is_returns = returns.iloc[window.train_start:window.train_end]
    is_res = run_strategy(cfg, is_prices, is_returns)
    is_m = compute_metrics(is_res.net_returns)
    is_sharpes.append(is_m.sharpe_ratio)
    
    # Out-of-sample
    oos_prices = prices.iloc[window.test_start:window.test_end]
    oos_returns = returns.iloc[window.test_start:window.test_end]
    oos_cfg = dataclasses.replace(
        cfg, backtest=dataclasses.replace(cfg.backtest, warmup_days=0)
    )
    oos_res = run_strategy(oos_cfg, oos_prices, oos_returns)
    oos_m = compute_metrics(oos_res.net_returns)
    oos_sharpes.append(oos_m.sharpe_ratio)
    
    print(f'Window {i+1}: IS Sharpe={is_m.sharpe_ratio:.2f}, OOS Sharpe={oos_m.sharpe_ratio:.2f}')

print(f'\nAvg IS Sharpe:  {np.mean(is_sharpes):.3f}')
print(f'Avg OOS Sharpe: {np.mean(oos_sharpes):.3f}')
print(f'OOS/IS ratio:   {np.mean(oos_sharpes)/np.mean(is_sharpes):.2f}')

In [ ]:
# 6. IS vs OOS Sharpe comparison
fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(is_sharpes))
width = 0.35
ax.bar(x - width/2, is_sharpes, width, label='In-Sample', color='steelblue')
ax.bar(x + width/2, oos_sharpes, width, label='Out-of-Sample', color='indianred')
ax.set_title('In-Sample vs Out-of-Sample Sharpe by Window')
ax.set_xlabel('Walk-Forward Window')
ax.set_ylabel('Sharpe Ratio')
ax.set_xticks(x)
ax.set_xticklabels([f'W{i+1}' for i in x])
ax.legend()
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Degrees of freedom count
print('\n=== Degrees of Freedom Audit ===')
dof = {
    'XSMOM lookbacks': '3 (63, 126, 252)',
    'XSMOM skip': '1 (21 days)',
    'TSMOM lookback': '1 (252)',
    'Bollinger window/std': '2 (20, 2.0)',
    'RSI window': '1 (14)',
    'Reversal window': '1 (5)',
    'Signal combination': '1 (equal weight)',
    'Portfolio construction': '1 (signal proportional)',
    'Position limit': '1 (5%)',
    'Leverage limit': '1 (1.0x)',
    'Fixed cost': '1 (5 bps)',
    'Impact cost': '1 (10 bps)',
    'Vol target': '1 (15%)',
    'DD threshold': '1 (15%)',
    'TOTAL DoF': '~17 parameters',
}
for k, v in dof.items():
    print(f'  {k}: {v}')
print('\nAll sourced from published academic literature.')